# Phase 1 & 2 : Chargement, Nettoyage des Données et Analyse Exploratoire (EDA)

Ce notebook présente la première phase du projet de prévision de séries temporelles du tourisme au Maroc. Nous allons charger, fusionner et nettoyer les données d'arrivées et de recettes touristiques, puis procéder à une analyse exploratoire de stationnarité, de décomposition et d'autocorrélation.

## Objectifs :
1. **Chargement et fusion** des données macroéconomiques et des arrivées/recettes.
2. **Nettoyage et Imputation** : Intégration des données COVID réelles de 2020-2021.
3. **Reconstruction historique** : Simulation mensuelle des arrivées et recettes (1996-2019) à partir des totaux annuels et des profils saisonniers récents.
4. **Analyse Exploratoire (EDA)** : Décomposition saisonnière (STL) et analyses d'autocorrélation (ACF/PACF).

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
import pandas as pd
import numpy as np
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller, kpss

# Ajout du dossier parent au chemin système pour importer src
sys.path.append(os.path.abspath('..'))

from src.config import DATA_DIR, TARGET_COL
import src.data_loader as loader
import src.cleaning as cleaner
import src.visualization as viz

## 1. Chargement et Fusion des Données

Nous chargeons les données macroéconomiques de base et les arrivées touristiques.

In [ ]:
merged_df = loader.load_and_merge_tourism_data()
print(f"Dimensions initiales : {merged_df.shape}")
merged_df.head()

## 2. Intégration des Données COVID Réelles (2020-2021)

Nous intégrons les données mensuelles réelles de la crise COVID pour corriger les valeurs aberrantes.

In [ ]:
merged_df = cleaner.integrate_covid_data(merged_df)
merged_df[(merged_df['Date'] >= '2020-03-01') & (merged_df['Date'] <= '2020-12-01')]

## 3. Reconstruction Historique Mensuelle (1996-2019)

Nous désagrégeons les totaux annuels historiques en données mensuelles réalistes à l'aide des profils saisonniers calculés sur la période récente (2022-2026).

In [ ]:
merged_df = cleaner.reconstruct_historical_arrivals(merged_df)
merged_df = cleaner.reconstruct_historical_receipts(merged_df)
print(f"Données manquantes Arrivals : {merged_df['Arrivals'].isna().sum()}")
print(f"Données manquantes Receipts : {merged_df['Total_Receipts_MDH'].isna().sum()}")

## 4. Sauvegarde du Dataset Nettoyé

In [ ]:
output_file = os.path.join(DATA_DIR, 'merged_tourism_data_final.csv')
merged_df.to_csv(output_file, index=False)
print(f"Sauvegardé avec succès -> {output_file}")

## 5. Analyse Exploratoire (EDA) & Graphiques

Nous visualisons l'évolution historique des arrivées et appliquons une décomposition STL.

In [ ]:
viz.plot_arrivals_evolution(merged_df, title="Évolution des Arrivées Touristiques au Maroc (1995-2026)")

In [ ]:
decomp_data = merged_df.set_index('Date')['Arrivals']
decomposition = seasonal_decompose(decomp_data, model='additive', period=12)
viz.plot_seasonal_decomposition(decomposition, title="Décomposition des Arrivées")

In [ ]:
viz.plot_acf_pacf(decomp_data, lags=24, title="Autocorrélation des Arrivées")

In [ ]:
corr_cols = ['REER', 'Oil_price', 'FDI', 'Poverty_rate', 'Arrivals', 'Total_Receipts_MDH', 'is_covid']
viz.plot_correlation_matrix(merged_df, corr_cols, title="Matrice de Corrélation des Variables Économiques")